# Notebook 4 — Model Training

**Input:** `X_train_sc.npy`, `X_test_sc.npy`, `y_train.npy`, `y_test.npy`

**Output:** `model_rf.pkl`, `model_xgb.pkl`, `model_lgbm.pkl`, `model_svm.pkl`

---

## Model strategy

| Model | Type | Class imbalance handling | Est. train time |
|---|---|---|---|
| Random Forest | Multi-class (7) | `class_weight='balanced'` | 3–5 min |
| XGBoost | Multi-class (7) | `sample_weight` from class counts | 2–4 min |
| LightGBM | Multi-class (7) | `is_unbalance=True` | 30–60 sec |
| SVM | **Binary only** (0/1) | `class_weight='balanced'` | 5–15 min |

> **Why SVM is binary only:** SVM with `kernel='rbf'` on 2M+ rows and 7 classes would take hours or run out of RAM. We use `LinearSVC` on binary labels instead.

---

## Install if needed
```bash
pip install xgboost lightgbm scikit-learn joblib
```

In [1]:
import numpy as np
import joblib
import time
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = r'D:\Computer Engineering 2022\4-CE.Z Forth Year\second-semester-4.2\project\ML-NIDS-Android\03-dataset\02-processed'

X_train = np.load(os.path.join(DATA_DIR, 'X_train_sc.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test_sc.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

# Binary labels for SVM
y_train_bin = (y_train > 0).astype(int)
y_test_bin  = (y_test  > 0).astype(int)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Classes : {np.unique(y_train)}')

X_train : (2262263, 52)
X_test  : (565566, 52)
Classes : [0 1 2 3 4 5 6]


In [2]:
# ── Helper: compute sample weights for XGBoost ───────────────────────────────
from sklearn.utils.class_weight import compute_sample_weight

def get_sample_weights(y):
    return compute_sample_weight(class_weight='balanced', y=y)

train_times = {}
models      = {}

In [3]:
# ── Model 1: Random Forest ────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

print('Training Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
train_times['RandomForest'] = time.time() - t0
models['RandomForest'] = rf
joblib.dump(rf, os.path.join(DATA_DIR, 'model_rf.pkl'))
print(f'Done in {train_times["RandomForest"]:.1f}s — saved model_rf.pkl')

Training Random Forest...
Done in 377.2s — saved model_rf.pkl


In [4]:
# ── Model 2: XGBoost ──────────────────────────────────────────────────────────
from xgboost import XGBClassifier

print('Training XGBoost...')
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=7,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(X_train, y_train, sample_weight=get_sample_weights(y_train))
train_times['XGBoost'] = time.time() - t0
models['XGBoost'] = xgb
joblib.dump(xgb, os.path.join(DATA_DIR, 'model_xgb.pkl'))
print(f'Done in {train_times["XGBoost"]:.1f}s — saved model_xgb.pkl')

Training XGBoost...
Done in 157.2s — saved model_xgb.pkl


In [5]:
# ── Model 3: LightGBM ─────────────────────────────────────────────────────────
from lightgbm import LGBMClassifier

print('Training LightGBM...')
t0 = time.time()
lgbm = LGBMClassifier(
    n_estimators=100,
    max_depth=-1,
    learning_rate=0.1,
    objective='multiclass',
    num_class=7,
    is_unbalance=True,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train, y_train)
train_times['LightGBM'] = time.time() - t0
models['LightGBM'] = lgbm
joblib.dump(lgbm, os.path.join(DATA_DIR, 'model_lgbm.pkl'))
print(f'Done in {train_times["LightGBM"]:.1f}s — saved model_lgbm.pkl')

Training LightGBM...
Done in 123.2s — saved model_lgbm.pkl


In [6]:
# ── Model 4: SVM (binary only) ────────────────────────────────────────────────
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print('Training SVM (binary, LinearSVC)...')
t0 = time.time()
svm_base = LinearSVC(
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)
# Wrap for probability estimates (needed for ROC-AUC)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_train, y_train_bin)
train_times['SVM'] = time.time() - t0
models['SVM'] = svm
joblib.dump(svm, os.path.join(DATA_DIR, 'model_svm.pkl'))
print(f'Done in {train_times["SVM"]:.1f}s — saved model_svm.pkl')

Training SVM (binary, LinearSVC)...
Done in 1931.3s — saved model_svm.pkl


In [7]:
# Summary of training times
print('\n=== Training Time Summary ===')
for name, t in train_times.items():
    mins = int(t // 60)
    secs = int(t % 60)
    print(f'  {name:<15}: {mins}m {secs:02d}s')
print('\nAll 4 models saved. Run Notebook 5 for evaluation.')


=== Training Time Summary ===
  RandomForest   : 6m 17s
  XGBoost        : 2m 37s
  LightGBM       : 2m 03s
  SVM            : 32m 11s

All 4 models saved. Run Notebook 5 for evaluation.
